# MLP crossover: [64, 64]

目标：固定类别数和 simplex 网格，只增加 MLP 的深度/宽度，观察单次梯度成本增大后，Adaptive Bundle + IPOPT 的梯度节省是否能抵消其外层优化开销。

近似时间模型：$T_B=N_BC_g+O_B$，$T_A=N_AC_g+O_I$。当 $(N_B-N_A)C_g>O_I-O_B$ 时，Adaptive 的墙钟时间可能优于 Baseline。每个 case 继续输出 `GN* vs time` 和 `GN* vs gradient evaluations` 两个 panel。

## 运行规则

1. Torch 与 cyipopt 在当前 macOS 环境有 OpenMP 加载顺序要求。请重启 kernel 后从本 notebook 第一格开始运行，不要提前 `import algorithm` 或 `import cyipopt`。
2. 本 notebook 只运行 `[64,64]`，可以与另外两个架构 notebook 并行。
3. 所有结果写入 `Adaptive Bundle Algorithm/output/complex_mlp_crossover/`。
4. 图中的 CPU 轴目前实际是 elapsed wall-clock time；本实验固定使用 CPU，不使用 GPU。

In [1]:
from pathlib import Path
import csv
import json
import sys

import matplotlib.pyplot as plt
import numpy as np

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd / 'Adaptive Bundle Algorithm' / 'Original_py',
    cwd / '.venv' / 'First-order-method-smooth-MOO' / 'Adaptive Bundle Algorithm' / 'Original_py',
]
ORIGINAL_PY = next((p for p in candidates if (p / 'experiments.py').exists()), None)
if ORIGINAL_PY is None:
    raise FileNotFoundError('Could not locate Adaptive Bundle Algorithm/Original_py')
if str(ORIGINAL_PY) not in sys.path:
    sys.path.insert(0, str(ORIGINAL_PY))

# experiments imports objectives_torch before algorithm/cyipopt.
from experiments import experiment_mlp_gn_coverage, mlp_parameter_count

OUTPUT_BASE = ORIGINAL_PY.parent / 'output' / 'complex_mlp_crossover'

print('Original_py:', ORIGINAL_PY)
print('Output base:', OUTPUT_BASE)

Original_py: /Users/shirch/vscode101/.venv/First-order-method-smooth-MOO/Adaptive Bundle Algorithm/Original_py
Output base: /Users/shirch/vscode101/.venv/First-order-method-smooth-MOO/Adaptive Bundle Algorithm/output/complex_mlp_crossover


In [2]:
# 只修改这一行：'quick'、'fast' 或 'full'。
RUN_MODE = 'fast'
PARALLEL_TAG = 'h64x64'

if RUN_MODE == 'quick':
    EXPECTED_RUNTIME = '约 10 秒'
    K, p, n = 2, 8, 40
    ARCHITECTURES = [[8], [12, 12]]
    SEEDS = [10]
    COARSE_RESOLUTION = 1
    N_PASSES = 1
    STEPS_PER_POINT = 1
    BASELINE_MAX_GRAD_EVALS = 4
    ADAPTIVE_MAX_GRAD_EVALS = 4
    EVAL_EVERY_N_GRADS = 2
    MAX_OUTER, MAX_INNER = 1, 1
    LAMBDA_MAX_STARTS = 2
    L_N_PROBES = 1
    ORACLE_BENCHMARK_REPEATS = 1
elif RUN_MODE == 'fast':
    EXPECTED_RUNTIME = '预计 20–60 分钟；Adaptive 达到 baseline 目标后会提前停止'
    # 三个受控宽度档位；除 hidden_sizes 外，其余实验条件完全相同。
    # 保持参数维度较小，避免 GN/IPOPT 被巨大梯度矩阵拖垮；
    # 用较大的样本数提高每次神经网络前向/反向传播的成本。
    K, p, n = 2, 20, 50_000
    ARCHITECTURES = [[64, 64]]
    SEEDS = [10]
    COARSE_RESOLUTION = 10
    N_PASSES = 1000  # max_grad_evals is the effective hard stop
    STEPS_PER_POINT = 5
    # K=2、resolution=10、每点5步：一个完整 baseline pass = 11*5*2 = 110 gradients。
    # Baseline 跑满 10 个 pass 定义共同 GN* 目标；Adaptive 使用 4 倍安全预算并在达标时停止。
    BASELINE_MAX_GRAD_EVALS = 1_100
    ADAPTIVE_MAX_GRAD_EVALS = 4_400
    EVAL_EVERY_N_GRADS = 110
    MAX_OUTER, MAX_INNER = 440, 5
    # K=2 时使用中心点和两个角点，避免单起点一直困在中心附近。
    LAMBDA_MAX_STARTS = 3
    L_N_PROBES = 1
    ORACLE_BENCHMARK_REPEATS = 1
elif RUN_MODE == 'full':
    EXPECTED_RUNTIME = '数小时到数天；不要用于试跑'
    # Hold K fixed so the first sweep isolates neural-network gradient cost.
    K, p, n = 5, 100, 1000
    ARCHITECTURES = [
        [32],
        [128, 128],
        [256, 256, 256],
        [512, 512, 512, 512],
    ]
    SEEDS = [10, 20, 30]
    COARSE_RESOLUTION = 5
    N_PASSES = 1000  # max_grad_evals is the effective hard stop
    STEPS_PER_POINT = 5
    BASELINE_MAX_GRAD_EVALS = 30_000
    ADAPTIVE_MAX_GRAD_EVALS = 30_000
    EVAL_EVERY_N_GRADS = 2_000
    MAX_OUTER, MAX_INNER = 2_000, 25
    LAMBDA_MAX_STARTS = 64
    L_N_PROBES = 5  # exploratory; increase for the final reported run
    ORACLE_BENCHMARK_REPEATS = 10
else:
    raise ValueError("RUN_MODE must be 'quick', 'fast', or 'full'")

OUTPUT_ROOT = OUTPUT_BASE / 'parallel' / PARALLEL_TAG
CURVE_DIR = OUTPUT_ROOT / 'curves'
HISTORY_DIR = OUTPUT_ROOT / 'histories'
for directory in (OUTPUT_ROOT, CURVE_DIR, HISTORY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('Mode:', RUN_MODE.upper())
print('Expected wall time:', EXPECTED_RUNTIME)
print('Run output:', OUTPUT_ROOT)
for arch in ARCHITECTURES:
    print(f'  hidden_sizes={arch}, d={mlp_parameter_count(K, p, arch):,}')

Mode: FAST
Expected wall time: 预计 20–60 分钟；Adaptive 达到 baseline 目标后会提前停止
Run output: /Users/shirch/vscode101/.venv/First-order-method-smooth-MOO/Adaptive Bundle Algorithm/output/complex_mlp_crossover/parallel/h64x64
  hidden_sizes=[64, 64], d=5,634


In [ ]:
def _json_ready(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'Not JSON serialisable: {type(value)!r}')

def _history_payload(result):
    payload = {}
    for key in ('baseline', 'algorithm2'):
        run = result[key]
        payload[key] = {
            name: run[name]
            for name in ('cov_history', 'grad_evals_history', 'cpu_times')
        }
    return payload

records = []
results = {}

for hidden_sizes in ARCHITECTURES:
    arch_tag = 'x'.join(map(str, hidden_sizes))
    for seed in SEEDS:
        case = f'K{K}_p{p}_n{n}_h{arch_tag}_seed{seed}'
        print('\n' + '=' * 80)
        print('Running', case)

        result = experiment_mlp_gn_coverage(
            verbose=True,
            K=K, p=p, n=n, hidden_sizes=hidden_sizes, seed=seed,
            init_seed=seed + 1,
            coarse_resolution=COARSE_RESOLUTION,
            n_passes=N_PASSES,
            steps_per_point_per_pass=STEPS_PER_POINT,
            eval_every_n_grads=EVAL_EVERY_N_GRADS,
            max_grad_evals=None,
            baseline_max_grad_evals=BASELINE_MAX_GRAD_EVALS,
            adaptive_max_grad_evals=ADAPTIVE_MAX_GRAD_EVALS,
            max_outer=MAX_OUTER, max_inner=MAX_INNER,
            lambda_max_starts=LAMBDA_MAX_STARTS,
            lambda_solver='ipopt', require_ipopt=True,
            prune_inner=False,
            l_n_probes=L_N_PROBES,
            oracle_benchmark_repeats=ORACLE_BENCHMARK_REPEATS,
            run_baseline=True, run_adaptive=True,
            out_path=str(CURVE_DIR / f'{case}.png'),
        )
        results[case] = result
        baseline = result['baseline']
        adaptive = result['algorithm2']
        target = float(baseline['cov_history'][-1])
        adaptive_final = float(adaptive['cov_history'][-1])
        reached = adaptive_final <= target
        baseline_time = float(baseline['cpu_times'][-1])
        adaptive_time = float(adaptive['cpu_times'][-1])
        baseline_grads = int(baseline['grad_evals_history'][-1])
        adaptive_grads = int(adaptive['grad_evals_history'][-1])

        record = {
            'case': case, 'seed': seed, 'K': K, 'p': p, 'n': n,
            'hidden_sizes': list(hidden_sizes), 'd': int(result['d']),
            'baseline_max_grad_evals': BASELINE_MAX_GRAD_EVALS,
            'adaptive_max_grad_evals': ADAPTIVE_MAX_GRAD_EVALS,
            'lambda_max_starts': LAMBDA_MAX_STARTS,
            'oracle_ms': float(result['oracle_ms']),
            'problem_setup_time': float(result['problem_setup_time']),
            'target_gn': target, 'adaptive_final_gn': adaptive_final,
            'adaptive_reached_target': bool(reached),
            'baseline_time': baseline_time, 'adaptive_time': adaptive_time,
            'baseline_grad_evals': baseline_grads,
            'adaptive_grad_evals': adaptive_grads,
            'time_ratio_baseline_over_adaptive': (
                baseline_time / adaptive_time if reached and adaptive_time > 0 else None
            ),
            'grad_ratio_baseline_over_adaptive': (
                baseline_grads / adaptive_grads if reached and adaptive_grads > 0 else None
            ),
            'curve_plot': result['plot'],
        }
        records.append(record)
        if reached:
            print(f'VALID comparison: both methods reached GN* <= {target:.4e}')
        else:
            print(
                f'INVALID comparison: Adaptive GN*={adaptive_final:.4e} '
                f'did not reach the Baseline target {target:.4e}; '
                'CPU ratio will not be reported.'
            )

        with (HISTORY_DIR / f'{case}.json').open('w', encoding='utf-8') as f:
            json.dump(_history_payload(result), f, indent=2, default=_json_ready)
        with (OUTPUT_ROOT / 'summary.json').open('w', encoding='utf-8') as f:
            json.dump(records, f, indent=2, default=_json_ready)

print(f'\nCompleted {len(records)} runs.')


Running K2_p20_n50000_h64x64_seed10
Coverage experiment — MLP (non-convex), metric = GN*
  K=2, p=20, n=50000, hidden_sizes=[64, 64], backend=torch, d=5634  |  L=[0.714 2.141] | init_seed=11 | setup=0.08s | oracle=30.76ms

  [uniform discretisation] ...
  Baseline pass 0/1000 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=5.5865e+01
  Baseline pass 1/1000 | t=1.81s | iters=55 | grad_evals=110 | worst-case pc=2.4048e-01
  Baseline pass 2/1000 | t=3.49s | iters=110 | grad_evals=220 | worst-case pc=1.1231e-01
  Baseline pass 3/1000 | t=5.16s | iters=165 | grad_evals=330 | worst-case pc=1.0923e-01
  Baseline pass 4/1000 | t=6.84s | iters=220 | grad_evals=440 | worst-case pc=1.0783e-01
  Baseline pass 5/1000 | t=8.48s | iters=275 | grad_evals=550 | worst-case pc=1.0556e-01
  Baseline pass 6/1000 | t=10.15s | iters=330 | grad_evals=660 | worst-case pc=1.0303e-01
  Baseline pass 7/1000 | t=11.82s | iters=385 | grad_evals=770 | worst-case pc=1.0033e-01
  Baseline pass 8/1000 | t=14.66s | 

In [ ]:
csv_path = OUTPUT_ROOT / 'summary.csv'
csv_fields = [
    'case', 'seed', 'K', 'p', 'n', 'hidden_sizes', 'd',
    'baseline_max_grad_evals', 'adaptive_max_grad_evals',
    'lambda_max_starts', 'oracle_ms',
    'problem_setup_time', 'target_gn', 'adaptive_final_gn',
    'adaptive_reached_target', 'baseline_time', 'adaptive_time',
    'baseline_grad_evals', 'adaptive_grad_evals',
    'time_ratio_baseline_over_adaptive',
    'grad_ratio_baseline_over_adaptive', 'curve_plot',
]
with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()
    for row in records:
        csv_row = dict(row)
        csv_row['hidden_sizes'] = 'x'.join(map(str, row['hidden_sizes']))
        writer.writerow(csv_row)

header = (
    f"{'architecture':>16} {'d':>10} {'oracle ms':>12} "
    f"{'BL time':>12} {'A time':>12} {'BL/A time':>12} {'reached':>9}"
)
print(header)
for row in records:
    ratio = row['time_ratio_baseline_over_adaptive']
    ratio_text = '-' if ratio is None else f'{ratio:.3f}'
    arch = 'x'.join(map(str, row['hidden_sizes']))
    print(
        f"{arch:>16} {row['d']:>10,d} {row['oracle_ms']:>12.3f} "
        f"{row['baseline_time']:>12.3f} {row['adaptive_time']:>12.3f} "
        f"{ratio_text:>12} {str(row['adaptive_reached_target']):>9}"
    )
print('CSV:', csv_path)

In [ ]:
from parallel_crossover_summary import plot_parallel_crossover

parallel_summary_plot = plot_parallel_crossover(OUTPUT_BASE)


## 解释结果

每个 case 曲线图的右侧 panel 是 `GN* vs gradient evaluations`。绿色虚线是 Baseline 最终 GN*；如果蓝色 Adaptive 曲线到达或低于绿色虚线，就表示 `adaptive_reached_target=True`。

- `Baseline / Adaptive wall time > 1`：Adaptive + IPOPT 更快。
- `Baseline / Adaptive gradient evaluations > 1`：Adaptive 使用更少梯度。
- 只有 `adaptive_reached_target=True` 的 case 才进入时间/梯度比值汇总。
- 如果梯度比始终大于 1、但时间比小于 1，说明 IPOPT/bundle 开销尚未被梯度成本覆盖。
- 本实现使用 ReLU，因此 $L_i$ 是经验估计；`L_N_PROBES=5` 适合探索，不应直接当作严格理论上界。

In [ ]:
print('Generated files:')
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file():
        print(' -', path.relative_to(OUTPUT_ROOT))